# Préparation de données et Nettoyage

### 1. Import des librairies

In [1]:
import pandas as pd
import numpy as np

### 2. Chargement des datasets

In [2]:
breast = pd.read_csv("C:\\Projet_filrouge\\oncobio_decision_analytics\\01_Data\\Interim\\SEER Breast Cancer Dataset .csv")
lung = pd.read_csv("C:\\Projet_filrouge\\oncobio_decision_analytics\\01_Data\\Processed\\ncctg_lung_patient_1000_sdv.csv")
biomarkers = pd.read_csv("C:\\Projet_filrouge\\oncobio_decision_analytics\\01_Data\\External_api\\biomarker_reference.csv")

print("Breast:", breast.shape)
print("Lung:", lung.shape)
print("Biomarkers:", biomarkers.shape)

Breast: (4024, 16)
Lung: (1228, 9)
Biomarkers: (10, 5)


### 3. Nettoyage Breast Cancer
#### 3.1 Harmonisation des noms

In [3]:
breast = breast.rename(columns={
    "Age": "age",
    "Survival Months": "survival_months",
    "Status": "event",
    "6th Stage": "stage"
})

#### 3.2 Nettoyage valeurs manquantes

In [4]:
breast = breast.dropna(subset=["age", "survival_months"])

#### 3.3 Encoder event en binaire

In [5]:
breast["event"] = breast["event"].map({
    "Alive": 0,
    "Dead": 1
})

#### 3.4 Ajouter cancer_type

In [6]:
breast["cancer_type"] = "Breast"

### 4. Nettoyage Lung Cancer
##### 4.1 Renommage

In [7]:
lung = lung.rename(columns={
    "TIME": "survival_months",
    "Y": "event"
})

#### 4.2 Sécurité valeurs

In [8]:
lung["survival_months"] = lung["survival_months"].clip(lower=1)
lung["event"] = lung["event"].astype(int)
lung["cancer_type"] = "Lung"

### 5. Harmonisation noyau commun

In [9]:
# On garde uniquement les colonnes d'intérêt
common_cols = [
    "age",
    "sex",
    "survival_months",
    "event",
    "cancer_type"
]
if "sex" not in breast.columns:
    breast["sex"] = "F"  # Breast dataset féminin uniquement
    
breast_common = breast[common_cols]
lung_common = lung[common_cols]

In [ ]:
breast_common.to_csv("C:\\Projet_filrouge\\oncobio_decision_analytics\\01_Data\\Processed\\breast_prepared.csv", index=False)
lung_common.to_csv("C:\\Projet_filrouge\\oncobio_decision_analytics\\01_Data\\Processed\\lung_prepared.csv", index=False)

### 6. Création table patient_master

In [10]:
patients_master = pd.concat(
    [breast_common, lung_common],
    ignore_index=True
)

patients_master["patient_id"] = range(1, len(patients_master)+1)

patients_master.head()

,age,sex,survival_months,event,cancer_type,patient_id
0,43,F,1,0,Breast,1
1,47,F,2,0,Breast,2
2,67,F,2,1,Breast,3
3,46,F,2,1,Breast,4
4,63,F,3,1,Breast,5


### 7. Création groupes de risque (valeur métier)

In [11]:
patients_master["risk_group"] = pd.cut(
    patients_master["survival_months"],
    bins=[0, 12, 36, 120],
    labels=["High Risk", "Medium Risk", "Low Risk"]
)

### 8. Création de quelques KPI 

In [12]:
kpi_summary = patients_master.groupby("cancer_type").agg({
    "patient_id": "count",
    "event": "mean",
    "survival_months": "median"
}).reset_index()

kpi_summary.rename(columns={
    "patient_id": "total_patients",
    "event": "mortality_rate",
    "survival_months": "median_survival"
}, inplace=True)

kpi_summary

,cancer_type,total_patients,mortality_rate,median_survival
0,Breast,4024,0.153082,73.0
1,Lung,1228,0.706840,267.5


### 9. Exportation final

In [14]:
patients_master.to_csv(
    r"C:\Projet_filrouge\oncobio_decision_analytics\01_Data\Processed\patients_master.csv",
    index=False
)